# Tutorial 07 - Character Prediction with LSTMs

This tutorial is based on the following two blog posts:
- http://karpathy.github.io/2015/05/21/rnn-effectiveness/
- https://towardsdatascience.com/writing-like-shakespeare-with-machine-learning-in-pytorch-d77f851d910c


The goal of this tutorial is to build a character level RNN and train it on a text corpus of some of Shakespeare's work. We will then use the trained RNN to hallucinate new text (one character at a time). 


![image info](mlp_vs_rnn.png)

![image info](rnn_flow.gif)

![image info](charseq.jpeg)

## Setup and Data Loading

We will use a small amount of Shakespears work to train the character prediction LSTM. Feel free to use different text data to train the network...

In [1]:
# Importing libraries
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F

Open the text file and show the first 100 characters. Feel free to experiment with your own text data of choice!

In [2]:
# Open shakespeare text file and read in data as `text`
with open('shakespeare.txt', 'r') as f:
    text = f.read()
    
# Showing the first 100 characters
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [3]:
len(text)

1115394

## Data Preparation


### 1- Map every character to one integer
We need to encode the characters in some way. We will simply map each unique character in the text to an integer (and vice versa).

In [4]:
# encoding the text and map each character to an integer and vice versa

# We create two dictionaries:
# 1. int2char, which maps integers to characters
# 2. char2int, which maps characters to integers
chars = tuple(set(text))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}

In [5]:
print(char2int)

{'E': 0, 'x': 1, 'o': 2, 'M': 3, '3': 4, 'W': 5, 'u': 6, 'y': 7, 'Y': 8, 'n': 9, 'a': 10, 'P': 11, ',': 12, 'O': 13, 'r': 14, 'L': 15, 'b': 16, "'": 17, 'F': 18, '\n': 19, 'v': 20, 'N': 21, 'Q': 22, 'X': 23, 'e': 24, 'g': 25, 'f': 26, 'q': 27, 't': 28, 'U': 29, 'k': 30, '$': 31, 'z': 32, 's': 33, 'l': 34, 'B': 35, 'D': 36, 'J': 37, 'S': 38, 'V': 39, 'G': 40, '?': 41, '-': 42, 'i': 43, 'w': 44, 'Z': 45, 'p': 46, 'd': 47, 'A': 48, ':': 49, ' ': 50, 'h': 51, '.': 52, 'T': 53, 'K': 54, '&': 55, ';': 56, 'm': 57, 'I': 58, '!': 59, 'j': 60, 'H': 61, 'C': 62, 'c': 63, 'R': 64}


In [6]:
print('Vocabulary size : ', len(chars))

Vocabulary size :  65


In [7]:
# Encode the text
encoded = np.array([char2int[ch] for ch in text])

# Showing the first 100 encoded characters
encoded[:100]

array([18, 43, 14, 33, 28, 50, 62, 43, 28, 43, 32, 24,  9, 49, 19, 35, 24,
       26,  2, 14, 24, 50, 44, 24, 50, 46, 14,  2, 63, 24, 24, 47, 50, 10,
        9,  7, 50, 26,  6, 14, 28, 51, 24, 14, 12, 50, 51, 24, 10, 14, 50,
       57, 24, 50, 33, 46, 24, 10, 30, 52, 19, 19, 48, 34, 34, 49, 19, 38,
       46, 24, 10, 30, 12, 50, 33, 46, 24, 10, 30, 52, 19, 19, 18, 43, 14,
       33, 28, 50, 62, 43, 28, 43, 32, 24,  9, 49, 19,  8,  2,  6])

### 2- One-hot-encoding

Let's also define some helper functions that map the ineger IDs to one-hot labels. The one-hot encodings will be used as inputs and targets for the network training (similar to multi-class classification).

In [8]:
# Defining method to encode one hot labels
##### arr (batch x seq_length) ----> (batch x seq_length x vocablary_size)
def one_hot_encode(arr, n_labels):
    
    # Initialize the the encoded array
    one_hot = torch.zeros(list(arr.shape) + [n_labels], dtype=torch.float32)
    one_hot = one_hot.view(arr.shape[0] * arr.shape[1], -1)
    
    # Fill the appropriate elements with ones
    one_hot[torch.arange(one_hot.shape[0]), arr.view(-1)] = 1.
    
    # Finally reshape it to get back to the original array
    one_hot = one_hot.reshape((*arr.shape, n_labels))
    
    return one_hot

In [9]:
one_hot_encode(arr=torch.LongTensor(np.array([[1, 2, 3, 0]])), n_labels=5)

tensor([[[0., 1., 0., 0., 0.],
         [0., 0., 1., 0., 0.],
         [0., 0., 0., 1., 0.],
         [1., 0., 0., 0., 0.]]])

### 3- Create dataset class for training

Finally, define a dataset class that will yield mini-batches for training. 

Suppose that our encoded training text is the following[2, 1, 4, 7, 0, 23, 57, 12]

1- split the text sequence into sub-sequences of lenth "seq_length=4"

[2, 1, 4, 7, 0, 23, 57, 12] ---------> [2, 1, 4, 7], [0, 23, 57, 12]

2- generate input sequence / target sequence for every sub-sequence

First input sequence: [2, 1, 4, 7] ---------> First output sequence: [1, 4, 7, 0] 

Second input sequence: [0, 23, 57, 12] ---------> Second output sequence: [23, 57, 12, 2]



In [10]:
class SubsequencesDataset(Dataset):
    def __init__(self, data: np.ndarray, seq_length: int):
        super(SubsequencesDataset, self).__init__()

        self.data = data
        self.seq_length = seq_length
        
    def __len__(self):
        if self.data.shape[0] % self.seq_length == 0:
            return self.data.shape[0] // self.seq_length - 1
        else:
            return self.data.shape[0] // self.seq_length

    def __getitem__(self, index: int):
        return (self.data[index * self.seq_length:index * self.seq_length + self.seq_length], 
                self.data[index * self.seq_length + 1:index * self.seq_length + self.seq_length + 1])

In [11]:
dataset = SubsequencesDataset(data=np.array([2, 1, 4, 7, 0, 23, 57, 12, 11]), seq_length=4)
len(dataset)

2

In [12]:
dataset[0]

(array([2, 1, 4, 7]), array([1, 4, 7, 0]))

In [13]:
dataset[1]

(array([ 0, 23, 57, 12]), array([23, 57, 12, 11]))

## Define the Recurrent Neural Network Model

First check if a GPU is available for training

In [14]:
# Check if GPU is available
train_on_gpu = torch.cuda.is_available()
if(train_on_gpu):
    print('Training on GPU!')
else: 
    print('No GPU available, training on CPU; consider making n_epochs very small.')

Training on GPU!


Now, it’s time to define the RNN network. We’re going to implement dropout for regularization and also create character dictionaries within the network. We’ll have LSTM units followed by a fully connected layer.

We’ll name it as Char-RNN because rather than having the input sequence be in words, we’re going to look at the individual letters/characters instead.

In the forward function, we’ll propagate the input and hidden state values through the LSTM layer to get the output and next hidden state values. After performing dropout, we reshape the output value to make it the proper dimensions for the fully connected layer.

Finally, we also have a section in the RNN for initializing the hidden state values for the correct batch size if using mini-batches.
![image info](lstm.png)

In [15]:
# Declaring the model
class CharRNN(nn.Module):
    
    def __init__(self, tokens, n_hidden=256, n_layers=2, drop_prob=0.5):
        super().__init__()
        
        self.drop_prob = drop_prob
        
        self.n_layers = n_layers
        
        self.n_hidden = n_hidden        
      
        self.chars = tokens
        
        #define the LSTM
        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layers, dropout=drop_prob, batch_first=True)
        
        #define a dropout layer
        self.dropout = nn.Dropout(drop_prob)
        
        #define the final, fully-connected output layer
        self.fc = nn.Linear(n_hidden, len(self.chars))
      
    
    def forward(self, x, hidden):
        ''' Forward pass through the network. 
            These inputs are x, and the hidden/cell state `hidden`. '''
                
        #get the outputs and the new hidden state from the lstm
        r_output, hidden = self.lstm(x, hidden)
        
        #pass through a dropout layer
        out = self.dropout(r_output)
        
        # Stack up LSTM outputs using view
        out = out.contiguous().view(-1, self.n_hidden)
        
        #put x through the fully-connected layer
        out = self.fc(out)
        
        # return the final output and the hidden state
        return out, hidden
    
    
    def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        # Create two new tensors with sizes n_layers x batch_size x n_hidden,
        # initialized to zero, for hidden state and cell state of LSTM
        weight = next(self.parameters()).data
        
        hidden = (weight.new_zeros(self.n_layers, batch_size, self.n_hidden),
                  weight.new_zeros(self.n_layers, batch_size, self.n_hidden))
        
        return hidden

## Training Code

Time for training! We declare a function, where we define an optimizer (Adam) and loss (cross entropy). We then create the training and validation data and initialize the hidden state of the RNN. 
We loop over the training set, each time encoding the data into one-hot vectors, performing forward and backpropagation, and updating the network parameters.

Every once a while, we generate some loss statistics (training loss and validation loss) to let us know if the model is training correctly.

In [16]:
# Declaring the train method
def train(net, data, epochs=10, batch_size=10, seq_length=50, lr=0.001, clip=5, val_frac=0.1, print_every=10):
    ''' Training a network 
    
        Arguments
        ---------
        
        net: CharRNN network
        data: text data to train the network
        epochs: Number of epochs to train
        batch_size: Number of mini-sequences per mini-batch, aka batch size
        seq_length: Number of character steps per mini-batch
        lr: learning rate
        clip: gradient clipping
        val_frac: Fraction of data to hold out for validation
        print_every: Number of steps for printing training and validation loss
    
    '''
    net.train()
    
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # create training and validation data
    dataset = SubsequencesDataset(data, seq_length=seq_length)
    train_size = int(len(dataset) * (1 - val_frac))
    val_size = len(dataset) - train_size
    training_set, validation_set = random_split(dataset, [train_size, val_size])

    # initialize dataloaders
    train_loader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(validation_set, batch_size=batch_size, shuffle=False)
    
    if(train_on_gpu):
        net.cuda()
    
    counter = 0
    n_chars = len(net.chars)
  
    for e in range(epochs):
        # initialize hidden state     
        
        for x, y in train_loader:
           
            counter += 1
            h = net.init_hidden(x.size(0))
            
            # One-hot encode our data and make them Torch tensors
            x = one_hot_encode(x, n_chars)            
            inputs, targets = x, y

            if(train_on_gpu):
                inputs, targets = inputs.cuda(), targets.cuda()
        
            # zero accumulated gradients
            net.zero_grad()
            
            # get the output from the model
            output, h = net(inputs, h)
            
            # calculate the loss and perform backprop
            loss = criterion(output, targets.view(x.size(0)*seq_length).long())
            loss.backward()
            
            # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            
            opt.step()
            
            # loss stats
            if counter % print_every == 0:
                with torch.no_grad():
                    # Get validation loss
                    val_losses = []
                    net.eval()
                    for x, y in val_loader:
                        # One-hot encode our data and make them Torch tensors
                        x = one_hot_encode(x, n_chars)

                        inputs, targets = x, y
                        if(train_on_gpu):
                            inputs, targets = inputs.cuda(), targets.cuda()

                        val_h = net.init_hidden(inputs.size(0))
                        output, val_h = net(inputs, val_h)
                        val_loss = criterion(output, targets.view(inputs.size(0)*seq_length).long())

                        val_losses.append(val_loss.item())

                    net.train() # reset to train mode after iterationg through validation data

                print("Epoch: {}/{}...".format(e+1, epochs),
                      "Step: {}...".format(counter),
                      "Loss: {:.4f}...".format(loss.item()),
                      "Val Loss: {:.4f}".format(np.mean(val_losses)))

Now, we just declare the hyperparameters for our model, create an instance for it, and train it!

In [17]:
# Define and print the net
n_hidden=512
n_layers=2

net = CharRNN(chars, n_hidden, n_layers)
print(net)

# Declaring the hyperparameters
batch_size = 128
seq_length = 100
n_epochs = 20 # start smaller if you are just testing initial behavior

# train the model
train(net, encoded, epochs=n_epochs, batch_size=batch_size, seq_length=seq_length, lr=0.001, print_every=50)

CharRNN(
  (lstm): LSTM(65, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=65, bias=True)
)
Epoch: 1/20... Step: 50... Loss: 3.3183... Val Loss: 3.3190
Epoch: 2/20... Step: 100... Loss: 3.2409... Val Loss: 3.2100
Epoch: 2/20... Step: 150... Loss: 2.7830... Val Loss: 2.7377
Epoch: 3/20... Step: 200... Loss: 2.4869... Val Loss: 2.4277
Epoch: 4/20... Step: 250... Loss: 2.3192... Val Loss: 2.2813
Epoch: 4/20... Step: 300... Loss: 2.2682... Val Loss: 2.1704
Epoch: 5/20... Step: 350... Loss: 2.1805... Val Loss: 2.0856
Epoch: 6/20... Step: 400... Loss: 2.0802... Val Loss: 2.0137
Epoch: 6/20... Step: 450... Loss: 2.0132... Val Loss: 1.9535
Epoch: 7/20... Step: 500... Loss: 2.0035... Val Loss: 1.9027
Epoch: 7/20... Step: 550... Loss: 1.9585... Val Loss: 1.8548
Epoch: 8/20... Step: 600... Loss: 1.9267... Val Loss: 1.8125
Epoch: 9/20... Step: 650... Loss: 1.8536... Val Loss: 1.7786
Epoch: 9/20... Step: 700.

## Generating new Text

After training, we create a method to predict the next character from the trained RNN with forward propagation.

In [18]:
# Defining a method to generate the next character
def predict(net, char, h=None, top_k=None):
        ''' Given a character, predict the next character.
            Returns the predicted character and the hidden state.
        '''
        
        # Char to int
        x = torch.LongTensor([[char2int[char]]])
        # Int to One-hot-encoding
        x = one_hot_encode(x, len(net.chars))
        inputs = x
        
        if(train_on_gpu):
            inputs = inputs.cuda()
        
        # get the output of the model
        out, h = net(inputs, h)

        # get the character probabilities
        p = F.softmax(out, dim=1).data
        if(train_on_gpu):
            p = p.cpu() # move to cpu
        
        # get top characters
        if top_k is None:
            top_ch = np.arange(len(net.chars))
        else:
            p, top_ch = p.topk(top_k)
            top_ch = top_ch.numpy().squeeze()
        
        # select the likely next character with some element of randomness
        p = p.numpy().squeeze()
        char = np.random.choice(top_ch, p=p/p.sum())
        
        # return the encoded value of the predicted char and the hidden state
        return int2char[char], h

Then, we define a sampling method that will use the previous method to generate an entire string of text, first using the characters in the first word and then using a loop to generate the next words using the top_k function, which chooses the letter with the highest probability to be next.

In [19]:
# Declaring a method to generate new text
def sample(net, size, prime='The', top_k=None):
        
    if(train_on_gpu):
        net.cuda()
    else:
        net.cpu()
    
    net.eval() # eval mode
    
    # First off, run through the prime characters
    chars = [ch for ch in prime]
    h = net.init_hidden(1)
    for ch in prime:
        char, h = predict(net, ch, h, top_k=top_k)

    chars.append(char)
    
    # Now pass in the previous character and get a new one
    for ii in range(size):
        char, h = predict(net, chars[-1], h, top_k=top_k)
        chars.append(char)

    return ''.join(chars)

Finally, just call the method, define the size you want (e.g., 1000 characters) and the starting character (e.g., ‘A’) and get the result:

In [20]:
# Generating new text
print(sample(net, 1000, prime='And', top_k=None))

And unto a spetty may make her,
Lice art the mist as boved where if the furght thy business;
Or, tendiment should fet will in thee for the
contunt
Geats.

PETRUCHIO:
Even of my groad from the meast!
Seach chould of cife, sir, dropp your higrne's forsure
As in day from thyself, to dalk you
What love thy hamber oner.
'queed, mort gisst my lord;
Even to our, as if my king to love.

CORIOLANUS:
We good all, how 'fish the iffairs of a soe,
That hould proformon assine not the nebling lack.
If it a prosected names didst very daich;
Bear fid, for stayn, which I an the neb? in course!
Nove what to unty that race, Hanring Awall:
Mide to any bage abept uncurniance foremanion,
But the shasticks wadly in bloody warrs
Jlovout and sibtely choused alfering with the spuck;
Who hateful stake the peace lost a'll plice begear,
Ahe that is not so. Ad in her gades to make
I she did very him not go comfurant
The darthing of my frounds in tumins youe
Our innoping growner's brother't child to cirton?
And what 